# Reproducing Harrison & Valavanis, *Quantum Wells, Wires and Dots* (4th ed.), with hetero1d

One representative result per chapter, computed with the **hetero1d** 1D heterostructure solver,
using the book's own material parameters (Appendix A) loaded as custom materials.
Each section compares a key number with the corresponding figure of the book.

Chapters 9 (3D quantum dots), 13 (optical waveguides), 14 (multiband k·p) and 15–16
(pseudopotentials) need methods beyond a single-band 1D envelope-function solver and are not included.

Source: https://github.com/erw-del/hetero1d

In [ ]:
!pip install -q git+https://github.com/erw-del/hetero1d.git

In [ ]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.optimize import brentq, minimize_scalar
from scipy.special import erf
from hetero1d import Structure, Solver, add_material, material
from hetero1d.core import CUSTOM, HB2M0, DOS0, KB

plt.rcParams.update({"figure.dpi": 120, "font.size": 9})
_trapz = getattr(np, "trapezoid", None) or np.trapz   # numpy 2 renamed trapz
K_C = 1.439964 / 13.18          # e^2/(4 pi eps0 eps_r) in eV nm  (eps_r = 13.18)
summary = []                    # (chapter, quantity, hetero1d, book)

# ---------------------------------------------------------------------------
# Book parameter set as hetero1d custom materials
# ---------------------------------------------------------------------------
def H(x, me=None, mhh=None, Vb=None):
    """Harrison's Ga(1-x)Al(x)As as a custom material (optionally override me, mhh, Vb)."""
    dEg = 1.247 * x
    Ec = 1.426 + (0.67 * dEg if Vb is None else Vb)
    name = f"H_AlGaAs_{x:.6f}_{me}_{mhh}_{Vb}"
    if name not in CUSTOM:
        add_material(name, Ec=Ec, Ev=-0.33 * dEg,
                     me=0.067 + 0.083 * x if me is None else me,
                     mhh=0.62 + 0.14 * x if mhh is None else mhh, mlh=0.087,
                     eps_s=13.18, eps_inf=10.89, hwLO=0.036, a=5.65 + 0.01 * x)
    return name


def bound_energy_analytic(Lw_nm, V0, mw, mb, n=0):
    """Exact BenDaniel-Duke finite-well level (even n=0, odd n=1), eV above well bottom."""
    def f(E):
        k = np.sqrt(mw * E / HB2M0); kap = np.sqrt(mb * (V0 - E) / HB2M0)
        return (k / mw * np.tan(k * Lw_nm / 2) - kap / mb) if n == 0 else \
               (k / mw / np.tan(k * Lw_nm / 2) + kap / mb)
    Emax = min(V0, HB2M0 / mw * (np.pi / Lw_nm) ** 2 * (1 if n == 0 else 4)) * (1 - 1e-9)
    Emin = 1e-9 if n == 0 else HB2M0 / mw * (np.pi / Lw_nm) ** 2 * 1.000001
    return brentq(f, Emin, min(Emax, V0 * (1 - 1e-9)))


def variational_binding(G, zr, dz, mass):
    """
    Max over lambda of  e^2/(4 pi eps) P/N - hbar^2/(2 m lambda^2)  for the 3D trial
    function chi * exp(-sqrt(r^2 + z^2)/lambda), G(z) = probability density in the
    relative coordinate z. Returns (binding energy eV, lambda nm). Used for donors
    (Ch. 5) and excitons (Ch. 6).
    """
    a = np.abs(zr)

    def E(lam):
        w = np.exp(-2 * a / lam)
        N = np.sum(G * w * (a + lam / 2)) * dz
        P = np.sum(G * w) * dz
        return HB2M0 / (mass * lam**2) - K_C * P / N

    r = minimize_scalar(E, bounds=(0.5, 60), method="bounded", options={"xatol": 1e-5})
    return -r.fun, r.x


def mean_lo_rate(sv, i, f, Te, TL, hw=0.036, eps_inf=10.89, eps_s=13.18, m=0.067):
    """
    Thermally averaged LO-phonon scattering rate i -> f (1/s), Froehlich interaction,
    bulk phonons, Maxwell-Boltzmann electrons at temperature Te in subband i,
    phonon occupation at lattice temperature TL; emission + absorption:
        W(k_i) = C (N + 1/2 -+ 1/2) * 1/(2 pi) Int dtheta A(q)/q ,
        C = m e^2 w (1/eps_inf - 1/eps_s) / (4 eps0 hbar^2),
        A(q) = Int Int psi_i psi_f(z) psi_i psi_f(z') exp(-q|z - z'|) dz dz'
    (reduces to the subband-minimum formula of hetero1d for k_i = 0).
    """
    hbar, e, m0, eps0, kB = 1.054571817e-34, 1.602176634e-19, 9.1093837e-31, 8.8541878128e-12, 1.380649e-23
    stride = max(1, len(sv.z) // 600)
    zs = sv.z[::stride] * 1e-9
    rho = sv.psi_e[::stride, i] * sv.psi_e[::stride, f] * 1e9          # 1/m
    dzs = (zs[1] - zs[0])
    D = np.abs(zs[:, None] - zs[None, :])
    qg = np.logspace(6, 10, 120)                                         # 1/m
    Ag = np.array([rho @ np.exp(-q * D) @ rho * dzs**2 for q in qg])
    A = lambda q: np.interp(np.log(q), np.log(qg), Ag)
    me = m * m0
    C = me * e**2 * (hw * e / hbar) * (1 / eps_inf - 1 / eps_s) / (4 * eps0 * hbar**2)
    N_LO = 1 / np.expm1(hw * e / (kB * TL))
    dE = (sv.Ee[i] - sv.Ee[f]) * e
    kTe = kB * Te
    Ek = np.linspace(0, 12 * kTe, 300)                                   # initial kinetic energy
    th = np.linspace(0, 2 * np.pi, 181)
    W = np.zeros_like(Ek)
    for sign, occ in ((+1, N_LO + 1), (-1, N_LO)):                       # emission, absorption
        Ekf = Ek + dE - sign * hw * e
        ok = Ekf > 0
        ki = np.sqrt(2 * me * Ek[ok]) / hbar
        kf = np.sqrt(2 * me * Ekf[ok]) / hbar
        q = np.sqrt(np.maximum(ki[:, None]**2 + kf[:, None]**2 - 2 * ki[:, None] * kf[:, None] * np.cos(th), 1e2))
        W[ok] += C * occ * _trapz(A(q) / q, th, axis=1) / (2 * np.pi)
    wgt = np.exp(-Ek / kTe)
    return _trapz(W * wgt, Ek) / _trapz(wgt, Ek)

## Chapter 1 -- heterostructures: band profiles, Type-I and Type-II alignment

Band alignment of heterostructures from the hetero1d material database: Type-I single/multiple wells and superlattices, and the Type-II (broken-gap) InAs/GaSb system.

In [ ]:
print("Chapter 1: band profiles")
fig, ax = plt.subplots(1, 4, figsize=(15, 3.6))
cases = [
    ("Single QW (Type I)", Structure.single_well(well="GaAs", barrier=("AlGaAs", 0.3), Lw=10, Lb=15)),
    ("Multiple QW", Structure.multi_well(4, well="GaAs", barrier=("AlGaAs", 0.3), Lw=6, Lb=10, Lb_out=15)),
    ("Superlattice", Structure.superlattice(8, well="GaAs", barrier=("AlGaAs", 0.3), Lw=4, Lb=2, Lb_out=10)),
    ("InAs/GaSb (Type II, broken gap)", Structure().add("GaSb", 15).add("InAs", 10).add("GaSb", 15)),
]
for a, (title, s) in zip(ax, cases):
    sv = Solver(s, T=300)
    a.plot(sv.z, sv.Ec, "b", label="E$_c$"); a.plot(sv.z, sv.Ev, "r", label="E$_v$")
    a.set_title(title); a.set_xlabel("z (nm)")
ax[0].set_ylabel("Energy (eV)"); ax[0].legend()
plt.tight_layout(); plt.savefig("ch01_band_profiles.png"); plt.close()
inas, gasb = material("InAs"), material("GaSb")
print(f"   InAs Ec = {inas['Ec']:.3f} eV lies below GaSb Ev = {gasb['Ev']:.3f} eV "
      f"-> broken gap of {(gasb['Ev']-inas['Ec'])*1000:.0f} meV")
summary.append(("1", "InAs/GaSb alignment", f"Type II, broken gap {(gasb['Ev']-inas['Ec'])*1000:.0f} meV",
                "Type II (Fig. 1.12)"))
from IPython.display import Image, display
display(Image("ch01_band_profiles.png"))


## Chapter 2 -- finite well, effective-mass mismatch (Fig. 2.19)

Ground-state energy of a finite GaAs well versus width for four barrier compositions, with constant and position-dependent effective mass. Every point is checked against the exact analytic BenDaniel–Duke solution.

In [ ]:
print("Chapter 2: E1 versus well width (Fig. 2.19)")
Lws = np.array([1, 2, 3, 4, 5, 6, 8, 10])          # nm
fig, a = plt.subplots(figsize=(5.5, 4))
maxerr = 0
for x, c in zip((0.1, 0.2, 0.3, 0.4), ("C0", "C1", "C2", "C3")):
    V0 = 0.67 * 1.247 * x
    for var_mass, mk, lab in ((True, "o", "m*(z)"), (False, "s", "const. m*")):
        Es, Ea = [], []
        for Lw in Lws:
            bar = H(x) if var_mass else H(x, me=0.067)
            s = Structure().add(bar, 40).add(H(0), Lw).add(bar, 40)   # long tails for weak binding
            sol = Solver(s, dz=0.02).solve(n_e=2, n_h=1)
            Es.append((sol.Ee[0] - sol.Ec.min()) * 1000)
            Ea.append(bound_energy_analytic(Lw, V0, 0.067, 0.067 + 0.083 * x if var_mass else 0.067) * 1000)
        maxerr = max(maxerr, np.max(np.abs(np.array(Es) - np.array(Ea))))
        a.plot(Lws * 10, Es, mk + ("-" if var_mass else "--"), color=c, mfc="w" if var_mass else c,
               label=f"x={x}" if var_mass else None)
a.set_xlabel("Well width (Å)"); a.set_ylabel("E$_1$ (meV)")
a.set_title("GaAs/Ga$_{1-x}$Al$_x$As: E$_1$ (open: m*(z), filled: constant m*)"); a.legend()
plt.tight_layout(); plt.savefig("ch02_E1_vs_width.png"); plt.close()
print(f"   max deviation from exact analytic solution: {maxerr:.3f} meV")
summary.append(("2", "E1 vs width, 4 alloys, 2 mass models", f"max error vs exact {maxerr:.3f} meV", "Fig. 2.19"))
from IPython.display import Image, display
display(Image("ch02_E1_vs_width.png"))


## Chapter 3 -- field-induced anti-crossing in an asymmetric double QW (Figs 3.24, 3.25)

Asymmetric double quantum well under an electric field: the two lowest states anti-cross where the wells come into resonance, and the wave functions swap wells.

In [ ]:
print("Chapter 3: field-induced anti-crossing (Figs 3.24-3.25)")
b = H(0.2)
dqw = Structure().add(b, 20).add(H(0), 6).add(b, 6).add(H(0), 5).add(b, 20)
Fs = np.linspace(0, 40, 81)
E1, E2, OV = [], [], []
for F in Fs:
    # hetero1d: positive field raises the right side; the book's field lowers the
    # right-hand (narrower, higher) well, i.e. negative field in hetero1d's convention
    sv = Solver(dqw, field=-F, dz=0.05).solve(n_e=6, n_h=1)
    i = sv.active_states(sv.psi_e)[:2]
    E1.append(sv.Ee[i[0]]); E2.append(sv.Ee[i[1]])
    OV.append(np.sum(np.abs(sv.psi_e[:, i[0]]) * np.abs(sv.psi_e[:, i[1]])) * sv.dz)
E1, E2, OV = np.array(E1), np.array(E2), np.array(OV)
k = np.argmin(E2 - E1)
ref = Solver(dqw).Ec.min()
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
ax[0].plot(Fs, (E1 - ref) * 1000, "o-", ms=3, label="state 1"); ax[0].plot(Fs, (E2 - ref) * 1000, "s-", ms=3, label="state 2")
ax[0].axvline(Fs[k], color="gray", ls=":"); ax[0].set_xlabel("Electric field (kV/cm)"); ax[0].set_ylabel("Energy (meV)")
ax[0].set_title("60 Å/60 Å/50 Å GaAs-Al$_{0.2}$Ga$_{0.8}$As DQW"); ax[0].legend()
ax[1].plot(Fs, OV, "k"); ax[1].axvline(Fs[k], color="gray", ls=":")
ax[1].set_xlabel("Electric field (kV/cm)"); ax[1].set_ylabel(r"$\int|\psi_1||\psi_2|dz$"); ax[1].set_title("Overlap of the two states")
plt.tight_layout(); plt.savefig("ch03_anticrossing.png"); plt.close()
print(f"   anti-crossing at F = {Fs[k]:.1f} kV/cm, minimum splitting {(E2-E1)[k]*1000:.2f} meV, max overlap {OV.max():.2f}")
summary.append(("3", "anti-crossing field", f"{Fs[k]:.1f} kV/cm", "10 kV/cm (Fig. 3.24)"))
from IPython.display import Image, display
display(Image("ch03_anticrossing.png"))


## Chapter 4 -- interdiffusion of a QW, constant D (Figs 4.6, 4.7)

Thermal interdiffusion of a quantum well (constant diffusion coefficient). The graded Al profile is built from 0.2 nm layers, showing that hetero1d handles arbitrary composition profiles.

In [ ]:
print("Chapter 4: diffused quantum well (Figs 4.6-4.7)")
D0 = 10e-2            # 10 A^2/s = 0.1 nm^2/s
x0, zl, zr, L = 0.1, 40.0, 60.0, 100.0     # nm; padded 200 A each side to mimic an infinite medium
slab = 0.2            # nm thickness of the thin layers representing the graded profile


def diffused_structure(t):
    s = Structure(name=f"t={t}s")
    zc = np.arange(slab / 2, L, slab)
    if t == 0:
        xs = np.where((zc > zl) & (zc < zr), 0.0, x0)
    else:
        w = 2 * np.sqrt(D0 * t)
        xs = x0 * (1 - 0.5 * (erf((zc - zl) / w) - erf((zc - zr) / w)))
    for xi in np.round(xs, 4):
        s.add(H(xi), slab)
    return s, zc, xs


ts = np.array([0, 10, 20, 50, 100, 200, 500, 1000])
Et = []
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for t in ts:
    s, zc, xs = diffused_structure(t)
    sv = Solver(s, dz=0.1).solve(n_e=1, n_h=1)
    Et.append((sv.Ee[0] - 1.426) * 1000)
    if t in (0, 10, 200, 500, 1000):
        ax[0].plot(zc * 10 - 200, xs, label=f"{t} s")
ax[0].set_xlim(0, 600); ax[0].set_xlabel("z (Å)"); ax[0].set_ylabel("Al fraction x"); ax[0].legend(fontsize=7)
ax[0].set_title("Diffusion, D$_0$ = 10 Å$^2$/s")
ax[1].plot(ts, Et, "o-"); ax[1].set_xlabel("Time (s)"); ax[1].set_ylabel("E$_1$ (meV above GaAs E$_c$)")
ax[1].set_title("Ground-state energy versus anneal time")
plt.tight_layout(); plt.savefig("ch04_diffusion.png"); plt.close()
print("   E1(t) =", ", ".join(f"{t}s:{e:.1f}" for t, e in zip(ts, Et)), "meV")
summary.append(("4", "E1 after 0 / 1000 s anneal", f"{Et[0]:.1f} / {Et[-1]:.1f} meV", "~5-10 / ~45 meV (Fig. 4.7)"))
from IPython.display import Image, display
display(Image("ch04_diffusion.png"))


## Chapter 5 -- donor binding energy versus well width (Figs 5.11, 5.12)

Binding energy of a donor at the well centre (3D variational trial function on top of the hetero1d electron wave function), including the approach to the bulk value in wide wells.

In [ ]:
print("Chapter 5: donor binding energy (Figs 5.11-5.12)")


def donor_binding(Lw, x, Lb=30.0):
    s = Structure().add(H(x, me=0.067), Lb).add(H(0), Lw).add(H(x, me=0.067), Lb)
    sv = Solver(s, dz=0.05).solve(n_e=1, n_h=1)
    chi2 = sv.psi_e[:, 0] ** 2
    zr = sv.z - sv.z[-1] / 2              # donor at the well centre
    return variational_binding(chi2, zr, sv.dz, 0.067)


fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
Lws = np.array([1, 2, 3, 4, 5, 6, 8, 10, 13, 16, 20])
peak = {}
for x in (0.1, 0.2, 0.3):
    Eb = [donor_binding(L_, x)[0] * 1000 for L_ in Lws]
    peak[x] = (max(Eb), Lws[int(np.argmax(Eb))] * 10)
    ax[0].plot(Lws * 10, Eb, "o-", ms=3, label=f"x = {x}")
ax[0].set_xlabel("Well width (Å)"); ax[0].set_ylabel("E$_{D^0}$ (meV)"); ax[0].set_title("Donor at well centre"); ax[0].legend()
Lw_wide = np.array([20, 30, 40, 60, 80, 100])
Ew = [donor_binding(L_, 0.1, Lb=40)[0] * 1000 for L_ in Lw_wide]
Ry = HB2M0 / 0.067 / (2 * HB2M0 / 0.067 / K_C) ** 2 * 1000 * 0 + K_C**2 * 0.067 / (4 * HB2M0) * 1000
ax[1].plot(Lw_wide * 10, Ew, "o-"); ax[1].axhline(Ry, ls="--", color="k", label=f"bulk {Ry:.2f} meV")
ax[1].set_xlabel("Well width (Å)"); ax[1].set_ylabel("E$_{D^0}$ (meV)"); ax[1].set_title("Wide wells, x = 0.1: approach to bulk"); ax[1].legend()
plt.tight_layout(); plt.savefig("ch05_donor.png"); plt.close()
print("   peaks:", ", ".join(f"x={x}: {p[0]:.1f} meV at {p[1]:.0f} A" for x, p in peak.items()),
      f"| 1000 A: {Ew[-1]:.2f} meV, bulk {Ry:.2f} meV")
summary.append(("5", "donor binding peak (x = 0.3)", f"{peak[0.3][0]:.1f} meV at {peak[0.3][1]:.0f} Å", "~12.5 meV at ~40 Å (Fig. 5.11)"))
from IPython.display import Image, display
display(Image("ch05_donor.png"))


## Chapter 6 -- exciton binding energy versus well width (Fig. 6.7)

Exciton binding energy versus well width, using hetero1d electron and heavy-hole wave functions in a variational calculation.

In [ ]:
print("Chapter 6: exciton binding energy (Fig. 6.7)")
mu = 0.067 * 0.62 / (0.067 + 0.62)
fig, a = plt.subplots(figsize=(5.5, 4))
ex = {}
for x, mk in ((0.1, "o"), (0.2, "s"), (0.3, "D")):
    EX = []
    for Lw in (2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20):
        s = Structure().add(H(x), 25).add(H(0), Lw).add(H(x), 25)
        sv = Solver(s, dz=0.05).solve(n_e=1, n_h=1)
        G = np.correlate(sv.psi_e[:, 0] ** 2, sv.psi_hh[:, 0] ** 2, mode="full") * sv.dz
        zr = (np.arange(len(G)) - (len(sv.z) - 1)) * sv.dz
        EX.append(-variational_binding(G, zr, sv.dz, mu)[0] * 1000)
    ex[x] = EX
    a.plot([2, 3, 4, 5, 6, 7, 8, 9, 10, 12, 14, 16, 18, 20], EX, "-" + mk, color="k", ms=4, label=f"x = {x}")
a.set_xticks([0, 5, 10, 15, 20]); a.set_xticklabels([0, 50, 100, 150, 200])
a.set_xlabel("Well width (Å)"); a.set_ylabel("E$_{X^0}$ (meV)"); a.set_title("Exciton in GaAs/Ga$_{1-x}$Al$_x$As QW"); a.legend()
plt.tight_layout(); plt.savefig("ch06_exciton.png"); plt.close()
print(f"   x=0.1: min {min(ex[0.1]):.2f} meV; 200 A: {ex[0.1][-1]:.2f} meV")
summary.append(("6", "exciton E_X0, x=0.1: minimum / 200 Å", f"{min(ex[0.1]):.2f} / {ex[0.1][-1]:.2f} meV", "~-9.4 / ~-7.5 meV (Fig. 6.7)"))
from IPython.display import Image, display
display(Image("ch06_exciton.png"))


## Chapter 7 -- piezoelectric field in GaN/AlN QWs (Fig. 7.14), via V_ext

Piezoelectric fields in GaN/AlN wells: step-linear built-in potential (field in the well, opposite field in the barriers) supplied through `V_ext`. Polarisation and barrier width are representative values, so the comparison is qualitative.

In [ ]:
print("Chapter 7: piezoelectric fields in GaN/AlN wells (Fig. 7.14)")
add_material("H_GaN", Ec=0.0, Ev=-3.2, me=0.2, mhh=1.0, mlh=0.2, eps_s=9.5)
add_material("H_AlN", Ec=2.0, Ev=-4.0, me=0.3, mhh=1.0, mlh=0.3, eps_s=8.5)
lb = 5.0                  # nm barriers (representative)
A_pol = 3.5               # MV/cm, (P_b - P_w)/eps  (representative, gives F_w ~ 5 MV/cm for 20 A wells)


def piezo_potential(lw):
    Fw = 2 * A_pol * lb / (lw + lb)          # MV/cm, eq. (7.67)
    Fb = -Fw * lw / (2 * lb)                 # eq. (7.69): Fw lw + 2 Fb lb = 0
    fw, fb = Fw * 0.1, Fb * 0.1              # eV/nm
    def V(z):
        return np.where(z < lb, fb * z,
               np.where(z < lb + lw, fb * lb + fw * (z - lb),
                        fb * lb + fw * lw + fb * (z - lb - lw)))
    return V, Fw


lws = np.array([1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5])
res = {True: ([], []), False: ([], [])}
for lw in lws:
    s = Structure().add("H_AlN", lb).add("H_GaN", lw).add("H_AlN", lb)
    for piezo in (False, True):
        V, Fw = piezo_potential(lw)
        sv = Solver(s, dz=0.01, V_ext=V if piezo else None).solve(n_e=2, n_h=1)
        res[piezo][0].append(sv.Ee[1] - sv.Ee[0])
        res[piezo][1].append(abs(np.sum(sv.psi_e[:, 0] * sv.z * sv.psi_e[:, 1]) * sv.dz) * 10)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
for piezo, mk, lab in ((False, "s-", "without piezo field"), (True, "o-", "with piezo field")):
    ax[0].plot(lws * 10, res[piezo][0], mk, label=lab); ax[1].plot(lws * 10, res[piezo][1], mk, label=lab)
ax[0].set_ylabel(r"$\Delta E_{12}$ (eV)"); ax[1].set_ylabel(r"|z$_{12}$| (Å)")
for a in ax: a.set_xlabel("Well width (Å)"); a.legend()
ax[0].set_title("GaN/AlN QW: transition energy"); ax[1].set_title("Dipole matrix element")
plt.tight_layout(); plt.savefig("ch07_piezo.png"); plt.close()
print(f"   50 A: dE12 = {res[False][0][-1]:.3f} eV (no field) -> {res[True][0][-1]:.3f} eV (field); "
      f"z12 = {res[False][1][-1]:.1f} -> {res[True][1][-1]:.1f} A")
summary.append(("7", "GaN/AlN 50 Å: dE12 without -> with field", f"{res[False][0][-1]:.2f} -> {res[True][0][-1]:.2f} eV",
                "~0.2 -> ~0.48 eV (Fig. 7.14, qualitative)"))
from IPython.display import Image, display
display(Image("ch07_piezo.png"))


## Chapter 8 -- finite square quantum wire, separable approximation (Fig. 8.9)

Square-cross-section GaAs quantum wire in the separable approximation E = E_y + E_z, from two 1D hetero1d calculations.

In [ ]:
print("Chapter 8: square quantum wire, E_yz = E_y + E_z (Fig. 8.9)")
sides = np.array([2, 3, 4, 5, 6, 8, 10, 15, 20, 25, 30])
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
wire = {}
for a, x in zip(ax, (0.2, 0.4)):
    e1, e2 = [], []
    for Ls in sides:
        s = Structure().add(H(x), 15).add(H(0), Ls).add(H(x), 15)
        sv = Solver(s, dz=0.05).solve(n_e=3, n_h=1)
        E = (sv.Ee - sv.Ec.min()) * 1000
        e1.append(E[0]); e2.append(E[1] if sv.e_bound[1] else np.nan)
    e1, e2 = np.array(e1), np.array(e2)
    wire[x] = 2 * e1[sides == 10][0]
    a.plot(sides * 10, 2 * e1, "o-", ms=3, label="1,1"); a.plot(sides * 10, e1 + e2, "s-", ms=3, label="1,2 / 2,1")
    a.plot(sides * 10, 2 * e2, "D-", ms=3, label="2,2")
    a.set_xlabel("Wire side (Å)"); a.set_ylabel("E$_{y,z}$ (meV)"); a.set_title(f"GaAs wire in Ga$_{{1-x}}$Al$_x$As, x = {x}"); a.legend()
plt.tight_layout(); plt.savefig("ch08_wire.png"); plt.close()
print(f"   100 A wire ground state: {wire[0.2]:.1f} meV (x=0.2), {wire[0.4]:.1f} meV (x=0.4)")
summary.append(("8", "100 Å square wire E_11 (x = 0.2 / 0.4)", f"{wire[0.2]:.0f} / {wire[0.4]:.0f} meV", "Fig. 8.9"))
from IPython.display import Image, display
display(Image("ch08_wire.png"))


## Chapter 10 -- LO-phonon intersubband scattering versus well width (Fig. 10.13)

Intersubband LO-phonon scattering 2→1 versus well width: rate from the subband minimum and the thermal mean over a 100 K electron distribution (Fröhlich interaction, emission and absorption).

In [ ]:
print("Chapter 10: LO-phonon 2->1 scattering versus well width (Fig. 10.13)")
widths = np.arange(10, 31, 1.0)
rates, Wmean, E21 = [], [], []
for Lw in widths:
    s = Structure().add(H(0.3), 6).add(H(0), Lw).add(H(0.3), 6)
    sv = Solver(s, T=15, dz=0.05).solve(n_e=3, n_h=1)
    E21.append((sv.Ee[1] - sv.Ee[0]) * 1000)
    r = [t["rate"] for t in sv.lo_phonon_rates() if t["i"] == 1 and t["f"] == 0]
    rates.append(r[0] if r else 0.0)
    Wmean.append(mean_lo_rate(sv, 1, 0, Te=100, TL=15))
rates, Wmean, E21 = np.array(rates), np.array(Wmean), np.array(E21)
cut = np.interp(36, E21[::-1], widths[::-1])
fig, a = plt.subplots(figsize=(5.5, 4))
a.plot(widths * 10, Wmean / 1e12, "o-", label="thermal mean, T$_e$ = 100 K")
a.plot(widths * 10, np.where(rates > 0, rates, np.nan) / 1e12, "--", color="gray", label="from subband minimum")
a.axvline(cut * 10, ls=":", color="gray")
a.set_ylim(0, 6); a.set_xlabel("Well width (Å)"); a.set_ylabel("W$_{21}$ (10$^{12}$ s$^{-1}$)")
a.set_title("LO-phonon 2$\\to$1, GaAs well, 60 Å Al$_{0.3}$GaAs barriers"); a.legend(fontsize=8)
plt.tight_layout(); plt.savefig("ch10_lo_phonon.png"); plt.close()
kp = int(np.argmax(Wmean))
print(f"   mean rate: 100 A {Wmean[0]:.2e} 1/s, peak {Wmean[kp]:.2e} 1/s at {widths[kp]*10:.0f} A, "
      f"300 A {Wmean[-1]:.2e} 1/s; E21 = E_LO at {cut*10:.0f} A")
summary.append(("10", "mean W21: 100 Å / peak", f"{Wmean[0]/1e12:.1f} / {Wmean[kp]/1e12:.1f}e12 s^-1 at {widths[kp]*10:.0f} Å",
                "~1.7 / ~4.3e12 s^-1 at ~175 Å (Fig. 10.13)"))
from IPython.display import Image, display
display(Image("ch10_lo_phonon.png"))


## Chapter 11 -- bound-bound intersubband absorption (Fig. 11.8)

Bound-to-bound intersubband absorption of an 8 nm GaAs/Al₀.₃₃Ga₀.₆₇As well at 77 K, Fermi level from the book.

In [ ]:
print("Chapter 11: intersubband absorption, 8 nm GaAs / Al0.33GaAs (Fig. 11.8)")
bar = H(0.33, me=0.094, Vb=0.286)
s = Structure().add(bar, 20).add(H(0), 8).add(bar, 20)
sv = Solver(s, T=77, dz=0.05).solve(n_e=4, n_h=1)
Ebar = 1.426 + 0.286
E1b, E2b = (sv.Ee[0] - Ebar) * 1000, (sv.Ee[1] - Ebar) * 1000
EF = Ebar - 0.230
kT = KB * 77
m1 = sv.mass_average(sv.psi_e[:, 0], sv.me)
n2d = sum(sv.mass_average(sv.psi_e[:, i], sv.me) * DOS0 * kT * np.logaddexp(0, (EF - sv.Ee[i]) / kT)
          for i in range(len(sv.Ee))) * 1e14
Eph = np.linspace(0.10, 0.15, 1000)
A = sv.isb_absorption(Eph, n2d_cm2=n2d, gamma=0.005) * 100
fig, a = plt.subplots(figsize=(5.5, 4))
a.plot(Eph * 1000, A, "k"); a.set_xlabel("Photon energy (meV)"); a.set_ylabel("Absorption per pass (%)")
a.set_title("8 nm GaAs/Al$_{0.33}$Ga$_{0.67}$As, 77 K, E$_F$ = -230 meV")
plt.tight_layout(); plt.savefig("ch11_absorption.png"); plt.close()
print(f"   E1 = {E1b:.0f} meV, E2 = {E2b:.0f} meV (rel. barrier), E21 = {E2b-E1b:.0f} meV, "
      f"n2D = {n2d:.2e} cm^-2, peak {A.max():.2f} % at {Eph[A.argmax()]*1000:.1f} meV")
summary.append(("11", "E1, E2 (rel. barrier); E21", f"{E1b:.0f}, {E2b:.0f}; {E2b-E1b:.0f} meV", "-243, -116; 127 meV (Fig. 11.8)"))
from IPython.display import Image, display
display(Image("ch11_absorption.png"))


## Chapter 12 -- quantum cascade laser active region (Figs 12.2, 12.4, 12.5)

Quantum cascade laser active region (triple well): subband energies and thermal-mean LO-phonon rates versus field, and the lifetime ratio that decides population inversion.

In [ ]:
print("Chapter 12: QCL triple-well active region (Figs 12.2, 12.4)")
b = H(0.2)
qcl = Structure().add(b, 20).add(H(0), 5.65).add(b, 5.65).add(H(0), 9.61).add(b, 2.825).add(H(0), 8.48).add(b, 20)
Fs = np.linspace(0, 20, 21)
Elev, W32, W21 = [], [], []
for F in Fs:
    # book convention: the field lowers the right-hand (downstream) side -> negative in hetero1d
    sv = Solver(qcl, T=77, field=-F, dz=0.025).solve(n_e=8, n_h=1)
    i = sv.active_states(sv.psi_e)[:3]
    Elev.append(sv.Ee[i])
    W32.append(mean_lo_rate(sv, i[2], i[1], Te=77, TL=77))
    W21.append(mean_lo_rate(sv, i[1], i[0], Te=77, TL=77))
Elev, W32, W21 = np.array(Elev), np.array(W32), np.array(W21)
ref = Solver(qcl).Ec.min()
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for j, mk in zip(range(3), ("o", "s", "D")):
    ax[0].plot(Fs, (Elev[:, j] - ref) * 1000, mk + "-", ms=3, label=f"E$_{j+1}$")
ax[0].set_xlabel("Field (kV/cm)"); ax[0].set_ylabel("Energy (meV)"); ax[0].set_title("Lowest three subbands"); ax[0].legend()
ax[1].plot(Fs, W21 / 1e10, "s-", ms=3, label="(2)-(1)"); ax[1].plot(Fs, W32 / 1e10, "o-", ms=3, label="(3)-(2)")
ax[1].set_xlabel("Field (kV/cm)"); ax[1].set_ylabel("Mean LO-phonon rate (10$^{10}$ s$^{-1}$)"); ax[1].legend()
ax[1].set_title("Thermal mean LO rates, 77 K")
ratio = W21 / W32
ax[2].plot(Fs, ratio, "k"); ax[2].axhline(1, ls=":", color="gray")
ax[2].set_xlabel("Field (kV/cm)"); ax[2].set_ylabel(r"$\tau_{32}/\tau_{21}$"); ax[2].set_title("Population inversion if > 1")
plt.tight_layout(); plt.savefig("ch12_qcl.png"); plt.close()
k10 = np.argmin(np.abs(Fs - 10))
print("   E1,E2,E3 at 0/10/20 kV/cm (meV):", [np.round((Elev[j]-ref)*1000,1).tolist() for j in (0, k10, -1)])
print(f"   at 10 kV/cm: E3-E2 = {(Elev[k10,2]-Elev[k10,1])*1000:.1f} meV, E2-E1 = {(Elev[k10,1]-Elev[k10,0])*1000:.1f} meV, "
      f"W21 = {W21[k10]:.2e}, W32 = {W32[k10]:.2e} 1/s, tau32/tau21 = {ratio[k10]:.2f}")
print("   W21 (1e10/s):", np.round(W21 / 1e10, 2)); print("   W32 (1e10/s):", np.round(W32 / 1e10, 2))
summary.append(("12", "QCL W21; W32 at 0 / 10 kV/cm (77 K)",
                f"{W21[0]/1e10:.1f} / {W21[k10]/1e10:.1f}; {W32[0]/1e10:.1f} / {W32[k10]/1e10:.1f} (1e10 s^-1)",
                "~3.5 / ~3.4; ~0.2 / ~3.1 (Fig. 12.4a)"))

# ===========================================================================
print("\n" + "=" * 100)
print(f"{'Ch':<4}{"Quantity":<42}{"hetero1d":<40}Book")
print("=" * 100)
for row in summary:
    print(f"{row[0]:<4}{row[1]:<42}{row[2]:<40}{row[3]}")
from IPython.display import Image, display
display(Image("ch12_qcl.png"))


## Summary: hetero1d versus the book

In [ ]:
print(f"{'Ch':<4}{'Quantity':<42}{'hetero1d':<40}Book")
for row in summary:
    print(f"{row[0]:<4}{row[1]:<42}{row[2]:<40}{row[3]}")